# 01 · Understand the real lab data
The original [Trinity College Dublin release](https://zenodo.org/records/10841412) contains graded exercise tests. Here **training_load is workload in watts**, and **performance_score is blood lactate in mmol/L**. These inherited CSV names are not long-term training load or competitive performance. One `athlete_id` identifies a *test file*, not necessarily a unique person.

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "data" / "processed" / "cycling_lactate.csv"
assert DATA.exists(), f"Dataset missing: {DATA}"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 20)
df = pd.read_csv(DATA, parse_dates=["date"])
print(df.head())

## Inspect coverage and quality
Each row is one measured stage. Count unique test files and verify missing values, date coverage, repeated rows, physical ranges and the number of stages per file.

In [ ]:
print("Rows:",len(df),"Test files:",df.athlete_id.nunique(),"Dates:",df.date.nunique())
print("Date range:",df.date.min().date(),"to",df.date.max().date())
print("Missing values:",df.isna().sum())
print("Exact duplicate rows:",df.duplicated().sum())
print("Stage counts by test file:",df.groupby("athlete_id").size().describe())
print(df[["training_load","performance_score"]].describe())

## Plot the observations
The relationship should bend upward as workload rises. Multiple lactate measurements from the same file are correlated, so a random **row** split would leak test-specific information.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].scatter(df.training_load,df.performance_score,s=12,alpha=.22)
axes[0].set(xlabel="Cycling workload (W)",ylabel="Blood lactate (mmol/L)",title="All measured stages")
counts=df.groupby(pd.cut(df.training_load,bins=np.arange(0,601,40)),observed=True).performance_score.agg(["median","count"])
centers=[iv.mid for iv in counts.index]
axes[1].plot(centers,counts["median"],marker="o")
axes[1].set(xlabel="Cycling workload (W)",ylabel="Median lactate (mmol/L)",title="Descriptive bins (not a model)")
plt.tight_layout();plt.show()
print(counts)

## Provenance and limits
`data/raw/graded_tests.zip` is the source archive. Run `python scripts/prepare_graded_tests.py` to reproduce this CSV and verify the source MD5. The summary workbook does not identify unique humans, so a person might have more than one test file. This dataset measures **acute exercise response**, not future race performance.